In [15]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code')

In [16]:
from copy import copy, deepcopy
import mwparserfromhell

import json
from birddog.core import (
    Archive, 
    )
from birddog.wiki import (
    ARCHIVE_BASE,
    WIKI_NAMESPACE,
    mw_read_page,
    mw_page_doc_url,
    get_title,
    check_page_changes,
    _expand_link_target,
    _read_wiki_text,
    _check_page_existence_chunked,
    _parse_wikitext_table,
    _is_table,
    )
from birddog.utility import fetch_url, form_text_item

In [ ]:
titles = [ 
    "Архів:ДАПО/978/1",
    "Архів:ДАЖО/1/74",
    "Архів:ЦДІАК/28/1",
    "Архів:ДАЖО/1",
    "Архів:ДАК/Р-352",
    "Архів:Архівний відділ виконавчого комітету Кременчуцької міської ради/Р",
    "Архів:ДАЖО/Д",
    "Архів:ДАДнО/Р-6478/2", 
    "Архів:ДАЖО/752", 
    "Архів:ДАКрО/225/1/25", 
    "Архів:ДАКрО/225", 
    "Архів:ДАСО/Р", 
    "Архів:ДАХмО/К", 
    "Архів:ДАКрО/225/1/144а", 
    "Архів:ДАПО/Р",
    "Архів:ДАХмО/Р-6193",
    "Архів:ДАКрО/П-5907/2Р",
    "Архів:ДАОО/Р-8085/1",
    "Архів:ДАКрО/225/1",
    "Архів:ДАПО/1072/1/1",
    "Архів:ДАПО/978/1/135",
    "Архів:ДАПО/978",
    ]

In [ ]:
pg = mw_read_page(titles[0])

In [ ]:
pg["header"]

In [ ]:
len(pg["children"])

In [ ]:
pg["children"][0]

In [ ]:
def save_page(title):
    fname = title.replace("/", "_")
    fname = f"./var/{fname}.json"
    with open(fname, "w") as file:
        file.write(json.dumps(mw_read_page(title)))

def load_page(title):
    fname = title.replace("/", "_")
    fname = f"./var/{fname}.json"
    with open(fname) as file:
        return json.loads(file.read())

In [ ]:
def itemize_links(link_dict, link_expander=None):
    def _form_item(x, key):
        expander = link_expander.get(key, lambda x: x)
        result = {"text": form_text_item(x)}
        result["link"] = expander(x)
        return result
    result = {}
    for key, value in link_dict.items():
        result[key] = [_form_item(item, key) for item in value]
    return result

In [ ]:
def itemize_page(pg):
    def _expand_target(target):
        return _expand_link_target(target, title)
    _expander = {
        "category_links": _expand_target,
        "internal_links": _expand_target,
        }
    for key in ["notes", "other_links"]:
        pg[key] = itemize_links(pg[key], link_expander=_expander)
        #print(f"{key}:\n {pg[key]}")
    return pg

In [ ]:
import difflib

def edit_distance(a, b):
    matcher = difflib.SequenceMatcher(None, a, b)
    return int(round((1 - matcher.ratio()) * max(len(a), len(b))))

def link_array_item_difference(a, b):
    return edit_distance(a["link"], b["link"])

def compare_dict_arrays(a, b, diff_fn, threshold):
    result = [None] * len(a)
    matched_b = [False] * len(b)

    # Pass 1: Exact matches
    for i in range(len(a)):
        if i < len(b) and diff_fn(a[i], b[i]) == 0:
            result[i] = "equal"
            matched_b[i] = True

    # Pass 2: Edits and adds for unmatched segments
    i, j = 0, 0
    while i < len(a):
        if result[i] == "equal":
            i += 1
            j += 1
            continue
        # Advance j to next unmatched b[j]
        while j < len(b) and matched_b[j]:
            j += 1

        if j >= len(b):
            result[i] = "added"
            i += 1
            continue
        delta = diff_fn(a[i], b[j])
        if delta <= threshold:
            result[i] = "edited"
            matched_b[j] = True
            i += 1
            j += 1
        else:
            result[i] = "added"
            i += 1
    return result

In [ ]:
def compare_page_link_arrays(page, reference):
    def _map_comparison(comp):
        return None if comp == 'equal' else comp
        
    for key in "notes", "other_links":
        if key in page:
            if key in reference:
                #print("--- comparing:", key)
                for target_array, ref_array in zip(page[key].values(), reference[key].values()):
                    comparison = compare_dict_arrays(target_array, ref_array, link_array_item_difference, 1)
                    comparison = [_map_comparison(comp) for comp in comparison]
                    #print("====>", target_array, comparison)
                    for value, comp in zip(target_array, comparison):
                        value["edit"] = comp
                        value["link_edit"] = comp
            else:
                for link_array in page[key].values():
                    #print(link_array)
                    for i, value in enumerate(link_array):
                        #print(value)
                        value["edit"] = "added"
                        value["link_edit"] = "added"

In [ ]:
for title in titles:
    print("----", title, "----")
    pg = itemize_page(mw_read_page(title))
    for key in ["notes", "other_links"]:
        print(f"--------- {key} ----:")
        print(pg[key])

In [ ]:
pg = load_page(titles[3])
pg1 = itemize_page(deepcopy(pg))
del pg["notes"]
del pg["other_links"]
#pgi

In [ ]:
#compare_page_link_arrays(pg1, pg)
#for key in ["notes", "other_links"]:
#    print(key, pgi[key])
#pgi

In [ ]:
pg2 = deepcopy(pg1)
pg2["notes"]["commons_links"][0]["link"] = "https://commons.wikimedia.org/wiki/File:ДАКрО_225-1-25_Про_здійснення_закладної_кріпості_Х.Г.Г.Ґрітберґа_та_Л.Н.Луценка._(1919).pdf"

In [ ]:
for key in ["notes", "other_links"]:
    print(key, pg2[key])
print("##########")
compare_page_link_arrays(pg2, pg1)
print("##########")
for key in ["notes", "other_links"]:
    print(key, pg2[key])

In [ ]:
pg1["other_links"]

In [ ]:
pg1["notes"]

In [ ]:
#pg = load_page(titles[0])
ref_pg = deepcopy(pg)
pg["description"] = form_text_item("changed description")
pg["children"][0][0]["link"] = "changed link"
pg["children"][0][1]["link"] = "new link"
pg["children"][1][1]["text"] = form_text_item("changed child description")
#print(pg["children"][1])
check_page_changes(pg, ref_pg, report=True)
#print(pg["children"][1])

In [ ]:
compare_dict_arrays(pg["other_links"]["category_links"], pg["other_links"]["category_links"], link_array_item_difference, 1)

In [ ]:
compare_dict_arrays(pg["other_links"]["category_links"], pg["notes"]["category_links"], link_array_item_difference, 1)

In [ ]:
archive = Archive('DAZHO', 'D')
print(archive.title, get_title(archive.title), get_title(archive.url))
fond = archive['1']
print(fond.title, get_title(fond.title), get_title(fond.url))
opus = fond['1']
print(opus.title, get_title(opus.title), get_title(opus.url))
case = opus['376']
print(case.title, get_title(case.title), get_title(case.url))


In [ ]:
get_title(archive.title)

In [ ]:
get_title(archive.url)

In [ ]:
archive.history(limit=5)

In [ ]:
get_title("%D0%90%D1%80%D1%85%D1%96%D0%B2%3A%D0%94%D0%90%D0%96%D0%9E/%D0%94")

In [ ]:
#read_page(opus.url)

In [ ]:
#mw_read_page(get_title(opus.url))

In [ ]:
#mw_read_page("Архів:ДАПО/978/1", oldid="748029")["lastmod"]

In [ ]:
#mw_read_page("Архів:ДАПО/978/1")["lastmod"]

In [ ]:
#current = mw_read_page("Архів:ДАПО/978/1")
#reference = mw_read_page("Архів:ДАПО/978/1", oldid="748029")
#check_page_changes(current, reference)

In [ ]:
for title in titles:
    print("---", title)
    save_page(title)

In [ ]:
import re 

url = "https://uk.wikisource.org/w/index.php?title=%D0%90%D1%80%D1%85%D1%96%D0%B2%3A%D0%94%D0%90%D0%96%D0%9E/%D0%94&oldid=632022"

match = re.search(r"[?&]oldid=(\d+)", url)
if match:
    oldid = match.group(1)
    print(oldid)  # Output: 632022

In [ ]:
pg = mw_read_page(titles[1])
pg["children"][0]

In [ ]:
pg["header"]

In [ ]:
pg["notes"]

In [ ]:
pg["description"]

In [ ]:
pg=mw_read_page(titles[1])
pg["children"][:5]

In [ ]:
pg=mw_read_page(titles[2])

In [ ]:
pg["children"]

In [ ]:
wt = _read_wiki_text(titles[0])

In [ ]:
def process_title(title):
    wikitext, revid, title = _read_wiki_text(title)
    wikicode = mwparserfromhell.parse(wikitext)

    # Table data
    tables = [t for t in wikicode.filter_tags() if _is_table(t)]
    header = []
    children = []
    all_page_links = set()

    if tables:
        table_code = tables[0].contents # assume first table
        print(table_code)
        header, children = _parse_wikitext_table(table_code)

    print(f"------ {title} ({len(children)} rows)")
    print(f"{ARCHIVE_BASE}/wiki/{WIKI_NAMESPACE}:{title.replace(' ', '_')}")
    print("header:", header)
    if children:
        print("child[0]:", children[0])

In [ ]:
process_title(titles[0])

In [ ]:
import re

def tokenize_wikitext_table_line(text):
    token_re = re.compile(r'''
        (\[\[.*?\]\])       |  # group 1: wikilink
        (\|\||\!\!)         |  # group 2: table cell separators
        ([^|\[\]!]+)           # group 3: everything else
    ''', re.VERBOSE)

    return [m.group(0) for m in token_re.finditer(text)]


In [ ]:
test1="|[[Архів:ДАПО/978/1/1|1]]||Алфавітний список рекрутів, прийнятих за 96-м рекрутським набором||4 березня - 14 квітня 1831||31||[https://www.familysearch.org/records/images/search-results?imageGroupNumbers=108149209  108149209]"


In [ ]:
tokenize_wikitext_table_line(test1)

In [6]:
pg = mw_read_page("https://uk.wikisource.org/wiki/Архів:ДААРК")

2025-09-23 14:20:24,851 [INFO] fetch_url: 1 requests in last 60s → 0.02 req/s


In [7]:
pg

{'title': {'uk': 'ДААРК'},
 'template': {'uk': 'заголовок'},
 'revid': 588157,
 'description': {'uk': 'Архіви'},
 'dates': {'uk': None, 'en': None},
 'notes': {'commons_links': [],
  'category_links': [],
  'internal_links': [],
  'external_links': ['https://archives.gov.ua/ua/%d0%b4%d0%b5%d1%80%d0%b6%d0%b0%d0%b2%d0%bd%d0%b8%d0%b9-%d0%b0%d1%80%d1%85%d1%96%d0%b2-%d0%b2-%d0%b0%d0%b2%d1%82%d0%be%d0%bd%d0%be%d0%bc%d0%bd%d1%96%d0%b9-%d1%80%d0%b5%d1%81%d0%bf%d1%83%d0%b1%d0%bb',
   'https://uk.wikipedia.org/wiki/%D0%94%D0%B5%D1%80%D0%B6%D0%B0%D0%B2%D0%BD%D0%B8%D0%B9_%D0%B0%D1%80%D1%85%D1%96%D0%B2_%D0%B2_%D0%90%D0%B2%D1%82%D0%BE%D0%BD%D0%BE%D0%BC%D0%BD%D1%96%D0%B9_%D0%A0%D0%B5%D1%81%D0%BF%D1%83%D0%B1%D0%BB%D1%96%D1%86%D1%96_%D0%9A%D1%80%D0%B8%D0%BC',
   'https://tsdea.archives.gov.ua/metric-books/?arch_id=11',
   'https://www.familysearch.org/records/images/search-results?page=1&place=5291']},
 'other_links': {'commons_links': [],
  'category_links': ['Категорія:Архіви України', 'Категорія:Кри

In [11]:
pg["tables"][0]["children"][0][0]["link"].replace("/wiki/","")

'Архів:ДААРК/Д'

In [13]:
pg = mw_read_page("https://uk.wikisource.org/wiki/%D0%90%D1%80%D1%85%D1%96%D0%B2:%D0%94%D0%90%D0%A1%D0%9E/%D0%A0-7720/14")

2025-09-23 15:38:13,824 [INFO] fetch_url: 1 requests in last 60s → 0.02 req/s


In [14]:
pg

{'title': {'uk': 'ДАСО/Р-7720/14'},
 'template': {'uk': 'Архіви/опис'},
 'revid': 906497,
 'description': {'uk': 'Середино-Будський район'},
 'dates': {'uk': '1918-1947', 'en': '1918-1947'},
 'notes': {'commons_links': ['https://commons.wikimedia.org/wiki/file:ДАСО_фонд_Р-7720_опис_14.pdf'],
  'category_links': [],
  'internal_links': [],
  'external_links': []},
 'other_links': {'commons_links': [],
  'category_links': ['Категорія:Середино-Будський район'],
  'internal_links': [],
  'external_links': []},
 'tables': [{'header': [{'uk': '№'},
    {'uk': 'Назва'},
    {'uk': 'РАЦС'},
    {'uk': 'Роки'},
    {'uk': 'Сторінки'}],
   'children': [[{'text': {'uk': '1', 'en': '1'},
      'link': '/wiki/Архів:ДАСО/Р-7720/14/1',
      'exists': True},
     {'text': {'uk': 'Книга реєстрації актів про шлюб'}, 'link': None},
     {'text': {'uk': 'БоровичіЖуравкаЗноб-НовгородськеКренидівкаКривоносівкаОчкинеСтягайлівкаСередина-БудаХильчичіЧернацьке'},
      'link': None},
     {'text': {'uk': '1918

In [18]:
pg = _read_wiki_text("https://uk.wikisource.org/wiki/%D0%A4%D0%B0%D0%B9%D0%BB:%D0%94%D0%90%D0%94%D0%BD%D0%9E_%D0%A4._%D0%A0-2567._%D0%9E._1._%D0%A1%D0%BF%D1%80._1.pdf")

2025-09-24 09:21:09,597 [INFO] fetch_url: 3 requests in last 60s → 0.05 req/s


RuntimeError: API error: {'code': 'invalidtitle', 'info': 'Bad title "https://uk.wikisource.org/wiki/%D0%A4%D0%B0%D0%B9%D0%BB:%D0%94%D0%90%D0%94%D0%BD%D0%9E_%D0%A4._%D0%A0-2567._%D0%9E._1._%D0%A1%D0%BF%D1%80._1.pdf".', '*': 'See https://uk.wikisource.org/w/api.php for API usage. Subscribe to the mediawiki-api-announce mailing list at &lt;https://lists.wikimedia.org/postorius/lists/mediawiki-api-announce.lists.wikimedia.org/&gt; for notice of API deprecations and breaking changes.'}